In [0]:
# Databricks notebook source
# ==============================================================================
# PHASE 7: Secure Authentication & Bronze Layer Ingestion
# ==============================================================================

storage_account_name = "datalakeseniorproj012026"
kv_scope_name = "kv-portfolio12026"

print("1. Authenticating via Azure Key Vault Secrets...")
try:
    client_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-id")
    tenant_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-tenant-id")
    client_secret = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-secret")
except Exception as e:
    print(f"Failed to retrieve secrets: {e}")
    raise

print("2. Injecting OAuth 2.0 configuration into Spark Session...")
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

raw_landing_uri = f"abfss://raw-landing@{storage_account_name}.dfs.core.windows.net/erp_export/"
bronze_uri = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"

print("3. Reading raw Parquet files from raw-landing zone...")
df_raw_lineitem = spark.read.parquet(raw_landing_uri + "lineitem/")
df_raw_orders = spark.read.parquet(raw_landing_uri + "orders/")

print("4. Appending metadata audit columns (Ingestion Timestamp & Source)...")
from pyspark.sql.functions import current_timestamp, lit

df_bronze_lineitem = df_raw_lineitem \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system", lit("ERP_TPCH_SF10"))

df_bronze_orders = df_raw_orders \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system", lit("ERP_TPCH_SF10"))

print("5. Writing 'lineitem' to Bronze Delta Lake...")
df_bronze_lineitem.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_uri + "lineitem/")

print("6. Writing 'orders' to Bronze Delta Lake...")
df_bronze_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_uri + "orders/")

print("==============================================================================")
print("SUCCESS: Bronze layer Delta tables successfully created and persisted.")
print("==============================================================================")

display(dbutils.fs.ls(bronze_uri))